# 階層ベイズMNL（対角共分散）による比較

対角共分散の階層ベイズMNL（HB-MNL）で camera 実データと PerSearch 疑似データを比較する。

記事の分析設計で採用している **bayesm::camera の生の符号化のまま**
（brand は4列ダミー・pixels/zoom/video/swivel/wifi は0/1・price は100ドル単位の
連続値、「どれも選ばない」は全列0の行として表現）に、対角共分散の階層構造
`beta_i ~ Normal(beta_bar, diag(sigma^2))` を適用する。

非中心化パラメータ化（`z ~ Normal(0,1)` を `sigma_beta` と組み合わせる）で収束を安定させる。
camera・PerSearch 両方に同一モデル・同一符号化を適用し、母集団平均 `beta_bar` と
個人間標準偏差 `sigma_beta` を比較する。

モデル定義・推定・要約のロジックは `conjoint_lib.hb` にまとまっている（`run_hb.py`・
`prior_sensitivity.py`・`predictive_check.py` からも同じ関数を呼んでいる）。

**使い方**: 下の「実行設定」セルで `DATASET` を `"camera"` または `"persearch"` に
切り替えてから、最後の実行セルを走らせる（1回の実行で1データセット分）。
notebook を開かず CLI から実行したい場合は `run_hb.py` を使う。


In [ ]:
from __future__ import annotations

import inspect

from conjoint_lib import schema
from conjoint_lib.hb import DEFAULT_PRIORS, DEFAULT_SEED, fit_hb_diag, run_hb


## モデル

記事の主題そのものなので、`conjoint_lib.hb.fit_hb_diag` の実装をそのまま表示する。

In [ ]:
print(inspect.getsource(fit_hb_diag))


## 実行設定

`DATASET` を書き換えてから実行セルを走らせる。`camera` は約2.4分、`persearch`（camera と同じ332体構成）は約4.7分
（12コア環境、C コンパイラ未導入・pytensor の numpy フォールバックのままの実測値。
課題数・収束の難しさにも左右されるため、目安として捉えること）。
`SEED` は事前分布・サンプリングの乱数シード（`conjoint_lib.hb.DEFAULT_SEED` と同じ既定値42を明示している。
値そのものは変えていない）。


In [ ]:
DATASET = "camera"  # "camera" または "persearch"
DRAWS = 1000
TUNE = 1000
CHAINS = 2
TARGET_ACCEPT = 0.9
SEED = DEFAULT_SEED
N_RESP_LIMIT = None  # 動作確認だけしたい場合は 20 などに絞る


## 実行

`r_hat` が1.01未満、`ess_bulk` が400超であることが理想。回答者数を camera と揃えた
現在のデータ（persearch 側は20〜50代に絞ったペルソナ）では chains=2・draws=1000 でも
camera 側 r_hat=1.005/ess_bulk=314、persearch 側 r_hat=1.022/ess_bulk=154 まで収束する
（beta_bar・sigma_beta 全パラメータのうち最大のr_hat・最小のess_bulk。回答者数が少ない
条件や persearch 側のデータ構成によってはもっと不安定になりやすいので注意）。
収束を上げたい場合は `draws`・`chains` を増やす（chains=4 が既定の推奨値）。


In [ ]:
run_hb(
    DATASET,
    draws=DRAWS, tune=TUNE, chains=CHAINS,
    target_accept=TARGET_ACCEPT, seed=SEED, n_resp_limit=N_RESP_LIMIT,
)
